In [1]:

from pathlib import Path
import sys
import importlib
import pandas as pd
import joblib

# 1. Project paths
PROJECT_ROOT = Path.cwd()

# If notebook is inside notebooks/, move one folder up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

WAREHOUSE_PATH = (
    PROJECT_ROOT
    / "for1sight_warehouse"
    / "prediction_training_v2_8_prerace.parquet"
)

MODEL_DIR = PROJECT_ROOT / "models" / "v2_8"

# 2. Load module
sys.path.insert(0, str(PROJECT_ROOT))

import future_race_predictor
importlib.reload(future_race_predictor)

from future_race_predictor import (
    FEATURE_COLUMNS,
    validate_warehouse,
    get_current_driver_roster,
    get_race_catalog,
    get_roster_as_of,
    build_future_feature_frame,
    predict_event
)

# 3. Load warehouse
warehouse = pd.read_parquet(WAREHOUSE_PATH)

validate_warehouse(warehouse)

print("Warehouse loaded:", warehouse.shape)

# 4. Load fixed 2026 roster
roster, roster_year = get_current_driver_roster(warehouse)

print("\n2026 Driver Roster")
print("Season:", roster_year)
print("Drivers:", len(roster))

# 5. Select a circuit for testing
race_catalog = get_race_catalog(warehouse)

selected_race = race_catalog.iloc[0]

print("\nTest Circuit:", selected_race["event_name"])
print("Event code:", selected_race["event_norm"])

# 6. Generate hypothetical 2027 features
grid_codes = roster["driver_code_norm"].tolist()

X_future = build_future_feature_frame(
    warehouse=warehouse,
    event_norm=selected_race["event_norm"],
    race_round=None,
    grid_codes=grid_codes,
    target_year=2027
)

print("\nFuture Feature Frame")
print("Shape:", X_future.shape)
print("Columns:", X_future.columns.tolist())

display(X_future.head())

# 7. Check feature completeness
print("\nMissing values:")
print(X_future.isna().sum())

assert X_future.shape == (22, 8)
assert X_future.columns.tolist() == FEATURE_COLUMNS

# 8. Load deployment models
model_files = {
    "Logistic Regression":
        "logistic_regression_deployment_v2_8.joblib",

    "Random Forest":
        "random_forest_deployment_v2_8.joblib",

    "Gradient Boosting":
        "gradient_boosting_deployment_v2_8.joblib"
}

models = {
    name: joblib.load(MODEL_DIR / filename)
    for name, filename in model_files.items()
}

# 9. Generate predictions
outputs = predict_event(
    models,
    X_future,
    grid_codes
)

for model_name, result in outputs.items():

    print(f"\n{model_name}")

    display(
        result[
            [
                "predicted_rank",
                "driver_code",
                "starting_grid",
                "race_win_probability"
            ]
        ].head(10)
    )

print("\n======================================")
print("V2.8 PREDICTION ENGINE TEST COMPLETED")
print("======================================")


Warehouse loaded: (1309, 26)

2026 Driver Roster
Season: 2026
Drivers: 22

Test Circuit: Abu Dhabi Grand Prix
Event code: abudhabi

Future Feature Frame
Shape: (22, 8)
Columns: ['starting_grid', 'driver_prior_avg_finish_3', 'driver_prior_avg_points_3', 'constructor_prior_avg_finish_3', 'constructor_prior_avg_points_3', 'driver_code_norm', 'constructor', 'event_norm']


,starting_grid,driver_prior_avg_finish_3,driver_prior_avg_points_3,constructor_prior_avg_finish_3,constructor_prior_avg_points_3,driver_code_norm,constructor,event_norm
0,1.0,17.666667,0.000000,16.000000,0.166667,ALB,Williams,abudhabi
1,2.0,18.000000,0.000000,19.500000,0.000000,ALO,Aston Martin,abudhabi
2,3.0,3.000000,16.666667,4.000000,14.500000,ANT,Mercedes,abudhabi
3,4.0,12.333333,0.666667,11.500000,1.000000,BEA,Haas F1 Team,abudhabi
4,5.0,16.666667,0.000000,13.166667,0.833333,BOR,Audi,abudhabi



Missing values:
starting_grid                     0
driver_prior_avg_finish_3         0
driver_prior_avg_points_3         0
constructor_prior_avg_finish_3    0
constructor_prior_avg_points_3    0
driver_code_norm                  0
constructor                       0
event_norm                        0
dtype: int64

Logistic Regression


,predicted_rank,driver_code,starting_grid,race_win_probability
0,1,ANT,3,0.747372
1,2,HAD,9,0.079388
2,3,LEC,13,0.034032
3,4,BEA,4,0.031583
4,5,ALB,1,0.025483
5,6,ALO,2,0.015675
6,7,VER,22,0.009325
7,8,BOR,5,0.008743
8,9,HAM,10,0.008209
9,10,HUL,11,0.007557



Random Forest


,predicted_rank,driver_code,starting_grid,race_win_probability
0,1,ANT,3,0.353010
1,2,VER,22,0.201839
2,3,LEC,13,0.099172
3,4,RUS,19,0.086238
4,5,HAD,9,0.066555
5,6,NOR,15,0.051454
6,7,ALB,1,0.037072
7,8,PIA,18,0.030489
8,9,ALO,2,0.029378
9,10,HAM,10,0.016917



Gradient Boosting


,predicted_rank,driver_code,starting_grid,race_win_probability
0,1,ALB,1,0.615447
1,2,ANT,3,0.221144
2,3,VER,22,0.063145
3,4,LEC,13,0.051539
4,5,ALO,2,0.029893
5,6,RUS,19,0.003757
6,7,BEA,4,0.002981
7,8,BOR,5,0.000984
8,9,BOT,6,0.000801
9,10,COL,7,0.000801



V2.8 PREDICTION ENGINE TEST COMPLETED
